

```
## **# RAG Node for Insurance Policy Question Answering**

This notebook builds a Retrieval-Augmented Generation (RAG) service over policy / benefit documents.

Includes:
- Place all policy `.txt` files in `./data/policies`
- Configure AI/ML API (e.g. OpenAI model via Opus AI/ML API)
- Build an index and expose the RAG via FastAPI
```



In [ ]:
 !pip install fastapi uvicorn openai python-dotenv pydantic[dotenv] requests

### **Setup & Importing**

In [ ]:
# !pip install fastapi uvicorn openai python-dotenv pydantic[dotenv] requests

import os
import glob
import json
from typing import List, Dict, Any


import numpy as np

from pydantic import BaseModel
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

# AI/ML API (e.g. OpenAI – this is what Opus AI/ML node will also call)
from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

### **Configuration**

In [ ]:
# === API keys & model names ===
OPENAI_API_KEY = "23e28ba122e4406cb8fabbec690566d0"
if not OPENAI_API_KEY:
    print("⚠️ Set OPENAI_API_KEY in your environment before running queries.")

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

# === Paths ===
DATA_DIR = "./data"
POLICY_DIR = os.path.join(DATA_DIR, "policies")
INDEX_DIR = os.path.join(DATA_DIR, "index")
os.makedirs(POLICY_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

INDEX_EMB_PATH = os.path.join(INDEX_DIR, "embeddings.npy")
INDEX_META_PATH = os.path.join(INDEX_DIR, "metadata.json")

### **Embedding + similarity helpers**

In [ ]:
def embed_texts(texts: List[str]) -> np.ndarray:
    """Call the AI/ML API to embed a batch of texts."""
    if client is None:
        raise RuntimeError("OpenAI client not configured. Set OPENAI_API_KEY.")
    resp = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    vectors = [d.embedding for d in resp.data]
    return np.array(vectors, dtype="float32")


def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between rows of a and b."""
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
    return a_norm @ b_norm.T

### **Load and Chunk Documents**

In [ ]:
import re

def load_policy_texts(policy_dir: str = POLICY_DIR) -> List[Dict[str, Any]]:
    """
    Load raw texts for each document.
    Assumes PDFs were converted to .txt with meaningful names:
      - PW-9582_Alami.txt
      - PW-9380_AbuDhabi.txt
      - Enaya_Beneficiary_Guide.txt
    """
    docs = []
    for path in glob.glob(os.path.join(policy_dir, "*.txt")):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        docs.append({
            "doc_id": os.path.basename(path),
            "source_path": path,
            "text": text,
        })
    print(f"Loaded {len(docs)} text documents from {policy_dir}")
    return docs


def chunk_text(text: str, max_tokens: int = 400, overlap: int = 50) -> List[str]:
    """Simple word-based chunking (good enough for hackathon)."""
    words = re.split(r"\s+", text)
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_tokens, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
        if start < 0:
            start = 0
    return chunks


def build_corpus_chunks(max_chunks: int = 200):
    chunks = []
    # example structure – keep your own logic inside
    for fname in os.listdir(POLICY_DIR):
        path = os.path.join(POLICY_DIR, fname)
        if not fname.endswith(".txt"):
            continue
        with open(path, "rb") as f:
           raw = f.read()

enc = chardet.detect(raw)["encoding"]
text = raw.decode(enc, errors="ignore")

        # whatever you use to split:
        for chunk_text in split_into_chunks(text):  # keep your own function
            chunks.append({
                "doc_id": fname,
                "text": chunk_text,
            })
            if len(chunks) >= max_chunks:
                return chunks  # 🔴 stop early

    return chunks

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 61)

### **Build and Save Index**

In [ ]:
def build_and_save_index():
    chunks = build_corpus_chunks(max_chunks=200)

    # TEMP LIMIT (safety cap)
    MAX_CHUNKS = 200
    chunks = chunks[:MAX_CHUNKS]

    texts = [c["text"] for c in chunks]

    BATCH_SIZE = 16
    all_embs = []

    print(f"Total chunks (capped): {len(texts)}")

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        print(f"Embedding batch {i}–{i + len(batch) - 1} ...")
        batch_emb = embed_texts(batch)
        all_embs.append(batch_emb)

    emb = np.vstack(all_embs)
    np.save(INDEX_EMB_PATH, emb)

    with open(INDEX_META_PATH, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print("✅ Index saved:", INDEX_EMB_PATH, INDEX_META_PATH)


def load_index():
    if not os.path.exists(INDEX_EMB_PATH) or not os.path.exists(INDEX_META_PATH):
        raise RuntimeError("Index not built yet. Run build_and_save_index() first.")
    emb = np.load(INDEX_EMB_PATH)
    with open(INDEX_META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return emb, meta

# IMPORTANT: run this once after you have the .txt files ready
# build_and_save_index()

### **Retrieval and Generation**

In [ ]:
def retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    emb, meta = load_index()
    q_vec = embed_texts([query])
    sims = cosine_sim_matrix(q_vec, emb)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        item = meta[i].copy()
        item["score"] = float(sims[i])
        results.append(item)
    return results


def build_system_prompt() -> str:
    return (
        "You are an insurance policy assistant. "
        "Answer based ONLY on the provided context from policy documents "
        "(ENAYA, Abu Dhabi policy wording, Alami policy, etc.). "
        "If you are not sure, say you are not sure and suggest contacting support. "
        "Always mention plan-specific limits / conditions if relevant."
    )


def generate_answer(query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    if client is None:
        raise RuntimeError("OpenAI client not configured.")

    context_text = "\n\n".join(
        f"[DOC: {c['doc_id']} – SCORE: {c['score']:.3f}]\n{c['text']}"
        for c in retrieved_chunks
    )

    messages = [
        {"role": "system", "content": build_system_prompt()},
        {
            "role": "user",
            "content": (
                "User question:\n"
                f"{query}\n\n"
                "Context from documents:\n"
                f"{context_text}"
            ),
        },
    ]

    completion = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.1,
    )
    return completion.choices[0].message.content.strip()


def rag_query(query: str, top_k: int = 5) -> Dict[str, Any]:
    retrieved = retrieve(query, top_k=top_k)
    answer = generate_answer(query, retrieved)
    return {
        "query": query,
        "answer": answer,
        "retrieved": retrieved,
    }

# Quick manual test (after index built):
# rag_query("Does ENAYA cover emergency overseas treatment?")

RAG Evaluation Testing

In [ ]:
from dataclasses import dataclass

@dataclass
class RAGTestCase:
    query: str
    must_contain: List[str]  # strings that should appear in answer+context

EVAL_TESTS = [
    RAGTestCase(
        query="Does ENAYA cover emergency medical treatment overseas?",
        must_contain=["emergency", "overseas", "ENAYA"],
    ),
    RAGTestCase(
        query="How are non-network claims reimbursed for Abu Dhabi Emirate plans?",
        must_contain=["reimbursement", "non-network", "Abu Dhabi"],
    ),
    # Add 3–5 more real questions from your docs
]

def eval_retrieval_hit_rate(tests: List[RAGTestCase]) -> None:
    emb, meta = load_index()
    total = len(tests)
    hit = 0
    for t in tests:
        retrieved = retrieve(t.query, top_k=5)
        text_block = " ".join(r["text"] for r in retrieved)
        if all(term.lower() in text_block.lower() for term in t.must_contain):
            hit += 1
    print(f"Retrieval hit rate: {hit}/{total} = {hit/total:.2f}")

def eval_end_to_end(tests: List[RAGTestCase]) -> None:
    total = len(tests)
    passed = 0
    for t in tests:
        out = rag_query(t.query, top_k=5)
        combined = (out["answer"] + " " +
                    " ".join(r["text"] for r in out["retrieved"]))
        ok = all(term.lower() in combined.lower() for term in t.must_contain)
        print("Q:", t.query)
        print("OK?:", ok)
        print("Answer snippet:", out["answer"][:250], "...")
        print("=" * 60)
        if ok:
            passed += 1
    print(f"E2E pass rate: {passed}/{total} = {passed/total:.2f}")

# Run after building index:
# eval_retrieval_hit_rate(EVAL_TESTS)
# eval_end_to_end(EVAL_TESTS)

Opus node integeration

In [ ]:
import requests

def opus_rag_node(input_payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    Example function representing an OPUS Python/HTTP node.

    Expected input_payload:
    {
        "user_question": "...",
        "plan_type": "ENAYA" | "ABU_DHABI" | null,
        "language": "en" | "ar" | null
    }
    """
    base_url = os.getenv("RAG_SERVICE_URL", "http://localhost:8000")
    question = input_payload.get("user_question", "")

    # Optional: enrich query with hints for better retrieval
    plan = input_payload.get("plan_type")
    lang = input_payload.get("language")
    if plan:
        question = f"[PLAN={plan}] " + question
    if lang:
        question = f"[LANG={lang}] " + question

    resp = requests.post(
        f"{base_url}/rag/query",
        json={"query": question, "top_k": 5},
        timeout=20,
    )
    resp.raise_for_status()
    data = resp.json()

    # Output back into Opus workflow
    return {
        "answer": data["answer"],
        "retrieved_chunks": data["retrieved"],
    }

OPUS_NODE_CONFIG_EXAMPLE = {
    "type": "python",
    "name": "RAG Policy QA Node",
    "inputs": ["user_question", "plan_type", "language"],
    "outputs": ["answer", "retrieved_chunks"],
    "handler": "opus_rag_node",
}

In [ ]:
build_and_save_index()

NameError: name 'build_and_save_index' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
result = rag_query("Does ENAYA cover overseas emergency treatment?")
print("ANSWER:\n", result["answer"])
print("\nNUMBER OF RETRIEVED CHUNKS:", len(result["retrieved"]))
print("\nFIRST CHUNK SNIPPET:\n", result["retrieved"][0]["text"][:300])

**FastAPI service (sample API endpoints)**

In [ ]:
app = FastAPI(title="RAG Policy Assistant")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class RAGQueryRequest(BaseModel):
    query: str
    top_k: int = 5

class RAGQueryResponse(BaseModel):
    query: str
    answer: str
    retrieved: List[Dict[str, Any]]

@app.get("/rag/health")
def health():
    return {"status": "ok"}

@app.post("/rag/query", response_model=RAGQueryResponse)
def rag_query_endpoint(body: RAGQueryRequest):
    out = rag_query(body.query, top_k=body.top_k)
    return out

@app.post("/rag/rebuild-index")
def rebuild_index():
    build_and_save_index()
    return {"status": "rebuilt"}

# Run API from terminal (same folder as this file):
# uvicorn app.rag_service:app --reload --port 8000

In [ ]:
# ================== IMPORTS ==================
import os
import glob
import json
import re
from typing import List, Dict, Any

import numpy as np
from openai import OpenAI


# ================== CONFIG ===================

# --- API keys & models ---
OPENAI_API_KEY = "YOUR_REAL_KEY_HERE"  # <-- put your key here

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

# --- Paths ---
DATA_DIR = "./data"
POLICY_DIR = os.path.join(DATA_DIR, "policies")
INDEX_DIR = os.path.join(DATA_DIR, "index")
os.makedirs(POLICY_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

INDEX_EMB_PATH = os.path.join(INDEX_DIR, "embeddings.npy")
INDEX_META_PATH = os.path.join(INDEX_DIR, "metadata.json")


# ================== HELPERS ==================

def safe_read_text(path: str) -> str:
    """
    Read a text file robustly without crashing on encoding issues.
    """
    # try utf-8 first
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except UnicodeDecodeError:
        # fallback to latin-1 and ignore errors
        with open(path, "rb") as f:
            raw = f.read()
        return raw.decode("latin-1", errors="ignore")


def load_policy_texts(policy_dir: str = POLICY_DIR) -> List[Dict[str, Any]]:
    """
    Load raw texts for each .txt document in policy_dir.
    """
    docs: List[Dict[str, Any]] = []
    for path in glob.glob(os.path.join(policy_dir, "*.txt")):
        text = safe_read_text(path)
        docs.append(
            {
                "doc_id": os.path.basename(path),
                "source_path": path,
                "text": text,
            }
        )
    print(f"Loaded {len(docs)} text documents from {policy_dir}")
    return docs


def chunk_text(text: str, max_tokens: int = 400, overlap: int = 50) -> List[str]:
    """
    Simple word-based chunking.
    """
    words = re.split(r"\s+", text)
    chunks: List[str] = []
    start = 0
    while start < len(words):
        end = min(start + max_tokens, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
        if start < 0:
            start = 0
    return chunks


def build_corpus_chunks(max_chunks: int = 400) -> List[Dict[str, Any]]:
    """
    Build a list of chunks from all documents, capped to max_chunks to avoid RAM issues.
    """
    chunks: List[Dict[str, Any]] = []
    docs = load_policy_texts(POLICY_DIR)

    for doc in docs:
        for ch in chunk_text(doc["text"]):
            chunks.append(
                {
                    "doc_id": doc["doc_id"],
                    "text": ch,
                }
            )
            if len(chunks) >= max_chunks:
                print(f"Reached max_chunks={max_chunks}, stopping.")
                return chunks
    print(f"Total chunks built: {len(chunks)}")
    return chunks


# ================== EMBEDDINGS ==================

def embed_texts(texts: List[str]) -> np.ndarray:
    """
    Embed a list of texts using OpenAI embeddings API.
    Returns a numpy array of shape (n, d).
    """
    if client is None:
        raise RuntimeError("OpenAI client not configured (API key missing).")

    resp = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    embs = [item.embedding for item in resp.data]
    return np.array(embs, dtype="float32")


def build_and_save_index(max_chunks: int = 400, batch_size: int = 16):
    """
    Build embeddings index in small batches to avoid RAM crash.
    Saves:
      - embeddings.npy
      - metadata.json
    """
    chunks = build_corpus_chunks(max_chunks=max_chunks)
    texts = [c["text"] for c in chunks]

    print(f"Indexing {len(texts)} chunks (batch_size={batch_size}) ...")
    all_embs: List[np.ndarray] = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        print(f"Embedding batch {i}–{i + len(batch) - 1} ...")
        batch_emb = embed_texts(batch)
        all_embs.append(batch_emb)

    if not all_embs:
        raise RuntimeError("No chunks to embed. Check your policies folder.")

    emb = np.vstack(all_embs)
    print("Final embeddings shape:", emb.shape)

    np.save(INDEX_EMB_PATH, emb)
    with open(INDEX_META_PATH, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print("✅ Index saved:", INDEX_EMB_PATH, INDEX_META_PATH)


def load_index():
    if not os.path.exists(INDEX_EMB_PATH) or not os.path.exists(INDEX_META_PATH):
        raise RuntimeError("Index not built yet. Run build_and_save_index() first.")
    emb = np.load(INDEX_EMB_PATH)
    with open(INDEX_META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return emb, meta


# ================== RETRIEVAL ==================

def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between each row in a and each row in b.
    a: (n, d)
    b: (m, d)
    """
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
    return a_norm @ b_norm.T


def retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    emb, meta = load_index()
    q_vec = embed_texts([query])
    sims = cosine_sim_matrix(q_vec, emb)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results: List[Dict[str, Any]] = []
    for i in top_idx:
        item = dict(meta[i])
        item["score"] = float(sims[i])
        results.append(item)
    return results


# ================== GENERATION ==================

def build_system_prompt() -> str:
    return (
        "You are an insurance policy assistant. "
        "Answer based ONLY on the provided context from policy documents "
        "(ENAYA, Abu Dhabi policy wording, Alami policy, etc.). "
        "If you are not sure, say you are not sure and suggest contacting support. "
        "Always mention plan-specific limits / conditions if relevant."
    )


def generate_answer(query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    if client is None:
        raise RuntimeError("OpenAI client not configured.")

    context_text = "\n\n".join(
        f"[DOC: {c['doc_id']} – SCORE: {c['score']:.3f}]\n{c['text']}"
        for c in retrieved_chunks
    )

    messages = [
        {"role": "system", "content": build_system_prompt()},
        {
            "role": "user",
            "content": (
                "User question:\n"
                f"{query}\n\n"
                "Context from documents:\n"
                f"{context_text}"
            ),
        },
    ]

    completion = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.1,
    )
    return completion.choices[0].message.content.strip()


def rag_query(query: str, top_k: int = 5) -> Dict[str, Any]:
    retrieved = retrieve(query, top_k=top_k)
    answer = generate_answer(query, retrieved)
    return {
        "query": query,
        "answer": answer,
        "retrieved": retrieved,
    }